In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

## Data Information

In [2]:
data = pd.read_csv('Streaming.csv')
data.head()

,Customer_ID,Age,Gender,Subscription_Length,Region,Payment_Method,Support_Tickets_Raised,Satisfaction_Score,Discount_Offered,Last_Activity,Monthly_Spend,Churned
0,CUST000001,56.0,Male,54,South,PayPal,0,9.0,6.42,319,62.11,1
1,CUST000002,69.0,Female,21,East,Debit Card,1,2.0,13.77,166,37.27,1
2,CUST000003,46.0,Female,49,East,PayPal,3,8.0,19.91,207,61.82,0
3,CUST000004,32.0,Male,47,West,Debit Card,3,1.0,13.39,108,40.96,1
4,CUST000005,60.0,Male,6,East,Credit Card,2,NaN,13.18,65,45.97,0


In [4]:
print("Data shape is : ", data.shape)

Data shape is :  (5000, 12)


In [5]:
print("Data columns are :")
print(data.columns)

Data columns are :
Index(['Customer_ID', 'Age', 'Gender', 'Subscription_Length', 'Region',
       'Payment_Method', 'Support_Tickets_Raised', 'Satisfaction_Score',
       'Discount_Offered', 'Last_Activity', 'Monthly_Spend', 'Churned'],
      dtype='object')


In [6]:
print('Missing values')
data.isnull().sum()

Missing values


Customer_ID                 0
Age                       500
Gender                      0
Subscription_Length         0
Region                      0
Payment_Method              0
Support_Tickets_Raised      0
Satisfaction_Score        500
Discount_Offered            0
Last_Activity               0
Monthly_Spend               0
Churned                     0
dtype: int64

In [7]:
print("Duplicate Values")
data.duplicated().sum()

Duplicate Values


0

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer_ID             5000 non-null   object 
 1   Age                     4500 non-null   float64
 2   Gender                  5000 non-null   object 
 3   Subscription_Length     5000 non-null   int64  
 4   Region                  5000 non-null   object 
 5   Payment_Method          5000 non-null   object 
 6   Support_Tickets_Raised  5000 non-null   int64  
 7   Satisfaction_Score      4500 non-null   float64
 8   Discount_Offered        5000 non-null   float64
 9   Last_Activity           5000 non-null   int64  
 10  Monthly_Spend           5000 non-null   float64
 11  Churned                 5000 non-null   int64  
dtypes: float64(4), int64(4), object(4)
memory usage: 468.9+ KB


In [9]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,4500.0,43.582222,14.969559,18.00,31.0000,43.000,56.00,69.00
Subscription_Length,5000.0,29.704400,17.050336,1.00,15.0000,29.000,44.00,59.00
Support_Tickets_Raised,5000.0,2.037000,1.422405,0.00,1.0000,2.000,3.00,9.00
Satisfaction_Score,4500.0,5.546000,2.869290,1.00,3.0000,6.000,8.00,10.00
Discount_Offered,5000.0,12.458404,4.325381,5.00,8.7000,12.500,16.19,20.00
Last_Activity,5000.0,181.441400,104.500951,1.00,90.0000,182.000,271.00,364.00
Monthly_Spend,5000.0,46.619784,9.522140,-5.06,39.8975,46.625,53.21,137.31
Churned,5000.0,0.448000,0.497338,0.00,0.0000,0.000,1.00,1.00


In [12]:
cat_cols = ['Gender', 'Region', 'Payment_Method']
num_cols = [col for col in data.columns if col not in cat_cols]
print('Num Cols: ', num_cols)
print('Cat cols: ', cat_cols)

Num Cols:  ['Customer_ID', 'Age', 'Subscription_Length', 'Support_Tickets_Raised', 'Satisfaction_Score', 'Discount_Offered', 'Last_Activity', 'Monthly_Spend', 'Churned']
Cat cols:  ['Gender', 'Region', 'Payment_Method']


In [16]:
print("Unique Values")
for col in cat_cols:
    print(data[col].value_counts())
    print('-----------------------------')

Unique Values
Gender
Female    2514
Male      2486
Name: count, dtype: int64
-----------------------------
Region
West     1316
North    1243
South    1239
East     1202
Name: count, dtype: int64
-----------------------------
Payment_Method
Debit Card     1697
PayPal         1673
Credit Card    1630
Name: count, dtype: int64
-----------------------------


In [17]:
data[data['Monthly_Spend'] < 0]

,Customer_ID,Age,Gender,Subscription_Length,Region,Payment_Method,Support_Tickets_Raised,Satisfaction_Score,Discount_Offered,Last_Activity,Monthly_Spend,Churned
1353,CUST001354,18.0,Male,33,South,PayPal,2,4.0,7.18,205,-5.06,0
3929,CUST003930,66.0,Female,10,East,Debit Card,1,2.0,14.65,140,-3.52,1


### Handle Missing Values

In [19]:
data['Age'] = data['Age'].fillna(data['Age'].median())

In [21]:
data['Satisfaction_Score'] = data['Satisfaction_Score'].fillna(data['Satisfaction_Score'].median())

### Handling Inconsistent Values

In [22]:
data['Monthly_Spend'] = data['Monthly_Spend'].apply(lambda x: abs(x) if x < 0 else x)

### Feature Engineering

#### Subscription Length to Categories

In [28]:
bins = [0, 15, 35, float('inf')]
labels = ['New', 'Mid Term', 'Loyal']

data['Tenure_Range'] = pd.cut(
        data['Subscription_Length'],
        bins=bins,
        labels=labels,
        right=False
)
data['Tenure_Range'].value_counts()

Tenure_Range
Loyal       2062
Mid Term    1735
New         1203
Name: count, dtype: int64

#### Activity level to Groups

In [29]:
bins = [0, 90, 180, 270, float('inf')]
labels = ['Recently Active', 'At Risk', 'Inactive', 'Dormant']

data['Activity_Level'] = pd.cut(
    data['Last_Activity'],
    bins=bins,
    labels=labels,
    right=False
)


In [31]:
data['Activity_Level'].value_counts()

Activity_Level
Inactive           1266
Dormant            1265
Recently Active    1244
At Risk            1225
Name: count, dtype: int64

#### Spending Segments

In [32]:
bins = [0, 40, 50, float('inf')]
labels = ['Low Spend', 'Medium Spend', 'High Spend']

data['Spend_Level'] = pd.cut(
    data['Monthly_Spend'],
    bins=bins,
    labels=labels,
    right=False
)

In [33]:
data['Spend_Level'].value_counts()

Spend_Level
High Spend      1868
Medium Spend    1862
Low Spend       1270
Name: count, dtype: int64

#### Support Tickets to Groups

In [36]:
bins = [0, 1, 3, float('inf')]
labels = ['No Issues', 'Few Issues(1-2)', 'Many Issues (3+)']

data['Issue_Group'] = pd.cut(
    data['Support_Tickets_Raised'],
    bins=bins,
    labels=labels,
    right=False
)

data['Issue_Group'].value_counts()

Issue_Group
Few Issues(1-2)     2743
Many Issues (3+)    1629
No Issues            628
Name: count, dtype: int64

#### Age to Groups

In [39]:
bins = [18, 35, 50, float('inf')]
labels = ['Young', 'Adult', 'Old']

data['AGe_Group'] = pd.cut(
    data['Age'],
    bins=bins,
    labels=labels,
    right=False
)

data['AGe_Group'].value_counts()

AGe_Group
Adult    1815
Old      1742
Young    1443
Name: count, dtype: int64

#### Satisfaction to Groups

In [42]:
bins = [1, 5, 8, 11]
labels = ['Low' ,'Neutral', 'High']

data['Satisfaction_Level'] = pd.cut(
    data['Satisfaction_Score'],
    bins=bins,
    labels=labels,
    right=False
)

In [43]:
data['Satisfaction_Level'].value_counts()

Satisfaction_Level
Neutral    1876
Low        1765
High       1359
Name: count, dtype: int64

#### Discount to Groups

In [44]:
bins = [5, 10, 15, 21]
labels = ['Low Discount', 'Medium Discount', 'High Discount']

data['Discount_Group'] = pd.cut(
    data['Discount_Offered'],
    bins=bins,
    labels=labels,
    right=False
)

data['Discount_Group'].value_counts()

Discount_Group
Low Discount       1692
Medium Discount    1663
High Discount      1645
Name: count, dtype: int64

### Hypothesis Testing

#### Test H1 : Inactivity Drives Churn

In [46]:
data['Churned'].value_counts(normalize=True)

Churned
0    0.552
1    0.448
Name: proportion, dtype: float64

In [49]:
pd.crosstab(data['Activity_Level'], data['Churned'], normalize='index')

Churned,0,1
Activity_Level,,
Recently Active,0.662379,0.337621
At Risk,0.680816,0.319184
Inactive,0.667457,0.332543
Dormant,0.203162,0.796838


##### Insights
The first 3 activity levels are similar, but the dormant level is got highest churn.

Dormant customers churn at 80% level which is double than other segments

Churn is not gradual

There is no point in return

Retention action must happen before dormancy

##### Recommendation:

The company should trigger re-engagemnet campaigns when users enter inactive stage

Do NOT wait until customer become dormancy

#### Test H2: Satisfaction drives churn

In [50]:
pd.crosstab(data['Satisfaction_Level'], data['Churned'], normalize='index')

Churned,0,1
Satisfaction_Level,,
Low,0.203966,0.796034
Neutral,0.718550,0.281450
High,0.774099,0.225901


##### Insights
Customers with low satisfaction churn at ~80%, while satisfied customers churn at only ~23%.

That is a 3.5× difference.

Churn is strongly tied to customer experience, not just engagement.

##### Recommendation:

Company should implement:

Early satisfaction surveys

Customer success outreach for low scores

Priority support for dissatisfied users


#### Test H3 : Support Issues Drives Churn

In [51]:
pd.crosstab(data['Issue_Group'], data['Churned'], normalize='index')

Churned,0,1
Issue_Group,,
No Issues,0.601911,0.398089
Few Issues(1-2),0.574553,0.425447
Many Issues (3+),0.494782,0.505218


##### Insights
Customers with many issues tend to churn with approximately 51%

Customers don’t churn just because they open tickets

But repeated issues gradually push them toward churn

##### Recommendation:

Identify Customer with 3+ tickets

Provide Priority support

Track Unresolved issues


#### Test H3 : Spend vs Churn

In [54]:
pd.crosstab(data['Spend_Level'], data['Churned'], normalize='index')

Churned,0,1
Spend_Level,,
Low Spend,0.204724,0.795276
Medium Spend,0.558539,0.441461
High Spend,0.781585,0.218415


##### Insights
Low spenders churn 3.6× more than high spenders.

##### Recommendation
Focus retention efforts on low-spend users

Create upgrade incentives

Improve perceived value of entry-level plans

#### Test H4: Tenure Group vs Churn Rate

In [55]:
pd.crosstab(data['Tenure_Range'], data['Churned'], normalize='index')

Churned,0,1
Tenure_Range,,
New,0.551953,0.448047
Mid Term,0.543516,0.456484
Loyal,0.559166,0.440834


##### Insights
Churn is almost same for every group


#### Test H5: Payment Method vs Churn

In [56]:
pd.crosstab(data['Payment_Method'], data['Churned'], normalize='index')

Churned,0,1
Payment_Method,,
Credit Card,0.554601,0.445399
Debit Card,0.556276,0.443724
PayPal,0.545129,0.454871


##### Insights

Billing friction is not a churn driver

#### Test H6: Age Group vs Churn

In [57]:
pd.crosstab(data['AGe_Group'], data['Churned'], normalize='index')

Churned,0,1
AGe_Group,,
Young,0.566875,0.433125
Adult,0.540496,0.459504
Old,0.551665,0.448335


#### Test H7: Gender vs Churn

In [58]:
pd.crosstab(data['Gender'], data['Churned'], normalize='index')

Churned,0,1
Gender,,
Female,0.554893,0.445107
Male,0.549075,0.450925


#### Test H8: Region vs Churn

In [59]:
pd.crosstab(data['Region'], data['Churned'], normalize='index')

Churned,0,1
Region,,
East,0.574875,0.425125
North,0.529364,0.470636
South,0.556901,0.443099
West,0.547872,0.452128


#### Test H9: Discount Group vs Churn

In [60]:
pd.crosstab(data['Discount_Group'], data['Churned'], normalize='index')

Churned,0,1
Discount_Group,,
Low Discount,0.561466,0.438534
Medium Discount,0.561034,0.438966
High Discount,0.533131,0.466869


#### Overall Insight

Strong churn drivers identified

Strength	Driver

🔴 Very strong -----> Dormant users

🔴 Very strong -----> Low satisfaction

🔴 Very strong -----> Low spend

🟠 Moderate    -----> Many support issues

Not churn drivers

Discounts,
Region,
Age,
Payment Method,
Tenure Group

### Churn Risk Segmentation

In [61]:
data['Risk_Inactive'] = data['Activity_Level'].isin(['Dormant', 'Inactive']).astype(int)
data['Risk_Low_Sat'] = (data['Satisfaction_Level'] == "Low").astype(int)
data['Risk_Low_Spend'] = (data['Spend_Level'] == "Low Spend").astype(int)
data['Risk_Many_Issues'] = (data['Issue_Group'] == "Many Issues (3+)").astype(int)

In [67]:
data['Risk_Score'] = data['Risk_Inactive'] + data['Risk_Low_Sat'] + data['Risk_Low_Spend'] + data['Risk_Many_Issues']

In [68]:
def segment_risk(risk_scores):
    if risk_scores <= 1:
        return 'Low Risk'
    elif risk_scores == 2:
        return 'Medium Risk'
    else:
        return "High Risk"

data['Risk_Level'] = data['Risk_Score'].apply(segment_risk)
data['Risk_Level'].value_counts()

Risk_Level
Low Risk       2881
Medium Risk    1288
High Risk       831
Name: count, dtype: int64

In [69]:
pd.crosstab(data['Risk_Level'], data['Churned'], normalize='index')

Churned,0,1
Risk_Level,,
High Risk,0.148014,0.851986
Low Risk,0.758417,0.241583
Medium Risk,0.350932,0.649068


In [70]:
data.to_csv("Streaming_Clean.csv", index=False)

In [71]:
data.head()

,Customer_ID,Age,Gender,Subscription_Length,Region,Payment_Method,Support_Tickets_Raised,Satisfaction_Score,Discount_Offered,Last_Activity,...,Issue_Group,AGe_Group,Satisfaction_Level,Discount_Group,Risk_Inactive,Risk_Low_Sat,Risk_Low_Spend,Risk_Many_Issues,Risk_Score,Risk_Level
0,CUST000001,56.0,Male,54,South,PayPal,0,9.0,6.42,319,...,No Issues,Old,High,Low Discount,1,0,0,0,1,Low Risk
1,CUST000002,69.0,Female,21,East,Debit Card,1,2.0,13.77,166,...,Few Issues(1-2),Old,Low,Medium Discount,0,1,1,0,2,Medium Risk
2,CUST000003,46.0,Female,49,East,PayPal,3,8.0,19.91,207,...,Many Issues (3+),Adult,High,High Discount,1,0,0,1,2,Medium Risk
3,CUST000004,32.0,Male,47,West,Debit Card,3,1.0,13.39,108,...,Many Issues (3+),Young,Low,Medium Discount,0,1,0,1,2,Medium Risk
4,CUST000005,60.0,Male,6,East,Credit Card,2,6.0,13.18,65,...,Few Issues(1-2),Old,Neutral,Medium Discount,0,0,0,0,0,Low Risk
